
# Пайплайн ранжирования обращений: LightGBM + CatBoost

Ноутбук решает задачу бинарного ранжирования обращений с основной метрикой **Daily Average Precision**.

Ключевые принципы:

1. Валидация выполняется по времени: последние даты train используются как validation.
2. Событийные признаки строятся только по событиям, для которых `event_ts < assignment_ts`.
3. `lead_id` не используется как модельный признак.
4. Категориальные признаки обрабатываются отдельно для CatBoost и LightGBM.
5. Итоговый score — ансамбль CatBoost и LightGBM с весом, выбранным на validation.
6. Все случайные состояния зафиксированы.


In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import random
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

warnings.filterwarnings("ignore")

random.seed(42)
np.random.seed(42)

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 200)

print("pandas:", pd.__version__)


pandas: 3.0.1


## 1. Конфигурация и загрузка данных

In [2]:
DATA_DIR = Path(".")

train_raw = pd.read_csv(DATA_DIR/'data/train.csv')
test_raw = pd.read_csv(DATA_DIR/'data/test.csv')
events_raw = pd.read_csv(DATA_DIR/'data/events.csv')
sample_submission = pd.read_csv(DATA_DIR/'sample_submission.csv')

print("train:", train_raw.shape)
print("test:", test_raw.shape)
print("events:", events_raw.shape)
print("sample:", sample_submission.shape)


train: (13694, 119)
test: (4306, 118)
events: (254705, 7)
sample: (4306, 2)


In [3]:

# Базовые проверки структуры
required_train = {"lead_id", "user_id", "assignment_ts", "assignment_date", "target"}
required_test = {"lead_id", "user_id", "assignment_ts", "assignment_date"}
required_events = {"lead_id", "user_id", "event_ts", "event_type"}

assert required_train.issubset(train_raw.columns)
assert required_test.issubset(test_raw.columns)
assert required_events.issubset(events_raw.columns)

assert train_raw["lead_id"].is_unique, "lead_id в train должен быть уникальным"
assert test_raw["lead_id"].is_unique, "lead_id в test должен быть уникальным"
assert set(train_raw["target"].dropna().unique()).issubset({0, 1})
assert len(sample_submission) == len(test_raw)

train_feature_cols = set(train_raw.columns) - {"target"}
test_feature_cols = set(test_raw.columns)
assert train_feature_cols == test_feature_cols, (
    f"Различия train/test: "
    f"только train={train_feature_cols-test_feature_cols}, "
    f"только test={test_feature_cols-train_feature_cols}"
)

print("Target rate:", train_raw["target"].mean())
print("Train dates:", train_raw["assignment_date"].min(), "—", train_raw["assignment_date"].max())
print("Test dates:", test_raw["assignment_date"].min(), "—", test_raw["assignment_date"].max())


Target rate: 0.20746312253541696
Train dates: 2026-04-07 — 2026-04-22
Test dates: 2026-04-23 — 2026-04-27


## 2.  Daily Average Precision

In [ ]:

def daily_average_precision(
    y_true,
    y_score,
    assignment_dates,
    zero_positive_policy: str = "zero",
):
    '''
    Для каждой даты отдельно считаем Average Precision,
    затем усредняем значения по датам.

    zero_positive_policy:
      - "zero": день без положительных объектов получает AP=0;
      - "skip": такой день исключается из среднего.

    В предоставленных данных желательно проверить, есть ли дни без target=1.
    '''
    metric_df = pd.DataFrame({
        "target": np.asarray(y_true, dtype=np.int8),
        "score": np.asarray(y_score, dtype=np.float64),
        "date": pd.to_datetime(assignment_dates).dt.normalize().to_numpy(),
    })

    rows = []
    for date, group in metric_df.groupby("date", sort=True):
        positives = int(group["target"].sum())

        if positives == 0:
            if zero_positive_policy == "skip":
                continue
            ap = 0.0
        else:
            ap = average_precision_score(group["target"], group["score"])

        rows.append({
            "date": date,
            "n_rows": len(group),
            "n_positive": positives,
            "positive_rate": group["target"].mean(),
            "average_precision": float(ap),
        })

    per_day = pd.DataFrame(rows)
    score = float(per_day["average_precision"].mean())
    return score, per_day


: 

## 3.  feature engineering

In [ ]:

WINDOWS = (1, 3, 7, 14, 30, 90)
EVENT_TYPES = sorted(events_raw["event_type"].dropna().astype(str).unique())

def safe_divide(numerator, denominator, smoothing: float = 1.0):
    return numerator / (denominator + smoothing)

def make_tabular_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Создаём признаки только из информации, находящейся в строке обращения.
    '''
    base = df.copy()

    base["assignment_ts"] = pd.to_datetime(base["assignment_ts"], errors="coerce")
    base["assignment_date"] = pd.to_datetime(base["assignment_date"], errors="coerce")

    ts = base["assignment_ts"]
    generated = {}

    # Календарные признаки
    generated["fe_assign_hour"] = ts.dt.hour.astype("float32")
    generated["fe_assign_minute"] = ts.dt.minute.astype("float32")
    generated["fe_assign_dow"] = ts.dt.dayofweek.astype("float32")
    generated["fe_assign_day"] = ts.dt.day.astype("float32")
    generated["fe_assign_weekofyear"] = ts.dt.isocalendar().week.astype("float32")
    generated["fe_minutes_from_midnight"] = (
        ts.dt.hour * 60 + ts.dt.minute
    ).astype("float32")

    generated["fe_hour_sin"] = np.sin(
        2 * np.pi * generated["fe_minutes_from_midnight"] / 1440
    ).astype("float32")
    generated["fe_hour_cos"] = np.cos(
        2 * np.pi * generated["fe_minutes_from_midnight"] / 1440
    ).astype("float32")
    generated["fe_dow_sin"] = np.sin(
        2 * np.pi * generated["fe_assign_dow"] / 7
    ).astype("float32")
    generated["fe_dow_cos"] = np.cos(
        2 * np.pi * generated["fe_assign_dow"] / 7
    ).astype("float32")

    # История назначений: конверсионные отношения
    for window in WINDOWS:
        assigned = f"leadgen_prev_assigned_{window}d"
        answered = f"leadgen_prev_answered_{window}d"
        positive = f"leadgen_prev_positive_{window}d"

        if {assigned, answered, positive}.issubset(base.columns):
            generated[f"fe_answer_rate_{window}d"] = safe_divide(
                base[answered], base[assigned]
            ).astype("float32")
            generated[f"fe_positive_per_answered_{window}d"] = safe_divide(
                base[positive], base[answered]
            ).astype("float32")
            generated[f"fe_positive_per_assigned_{window}d"] = safe_divide(
                base[positive], base[assigned]
            ).astype("float32")

    # Автоматическое распознавание признаков с окнами *_1d, *_3d и тд
    pattern = re.compile(r"(.+)_(1|3|7|14|30|90)d$")
    window_groups = {}

    for column in base.columns:
        match = pattern.fullmatch(column)
        if match and pd.api.types.is_numeric_dtype(base[column]):
            feature_name = match.group(1)
            window = int(match.group(2))
            window_groups.setdefault(feature_name, {})[window] = column

    for feature_name, columns_by_window in window_groups.items():
        # Нормализация на длину окна
        for window, column in columns_by_window.items():
            generated[f"fe_{feature_name}_per_day_{window}d"] = (
                base[column] / float(window)
            ).astype("float32")

        # Недавняя активность относительно более длинной истории
        if 1 in columns_by_window and 7 in columns_by_window:
            col_1 = columns_by_window[1]
            col_7 = columns_by_window[7]
            generated[f"fe_{feature_name}_ratio_1d_7d"] = safe_divide(
                base[col_1], base[col_7]
            ).astype("float32")
            generated[f"fe_{feature_name}_lift_1d_7d"] = (
                base[col_1] - base[col_7] / 7.0
            ).astype("float32")

        if 7 in columns_by_window and 30 in columns_by_window:
            col_7 = columns_by_window[7]
            col_30 = columns_by_window[30]
            generated[f"fe_{feature_name}_ratio_7d_30d"] = safe_divide(
                base[col_7], base[col_30]
            ).astype("float32")
            generated[f"fe_{feature_name}_lift_7d_30d"] = (
                base[col_7] / 7.0 - base[col_30] / 30.0
            ).astype("float32")

    generated_df = pd.DataFrame(generated, index=base.index)
    result = pd.concat([base, generated_df], axis=1)

    result = result.replace([np.inf, -np.inf], np.nan)
    return result


: 

In [ ]:

def make_event_features(
    assignments: pd.DataFrame,
    events: pd.DataFrame,
    windows=WINDOWS,
    event_types=EVENT_TYPES,
) -> pd.DataFrame:
    '''
    Строим events признаки без утечки.
    event_ts < assignment_ts

    Поскольку lead_id уникален в train/test, каждое событие связывается
    с единственным моментом назначения соответствующего обращения.
    '''
    assignment_frame = assignments[["lead_id", "assignment_ts"]].copy()
    assignment_frame["assignment_ts"] = pd.to_datetime(
        assignment_frame["assignment_ts"], errors="coerce"
    )

    event_frame = events.copy()
    event_frame["event_ts"] = pd.to_datetime(
        event_frame["event_ts"], errors="coerce"
    )
    event_frame["event_type"] = event_frame["event_type"].astype(str)

    merged = event_frame.merge(
        assignment_frame,
        on="lead_id",
        how="inner",
        validate="many_to_one",
    )

    before_filter = len(merged)
    merged = merged.loc[
        merged["event_ts"].notna()
        & merged["assignment_ts"].notna()
        & (merged["event_ts"] < merged["assignment_ts"])
    ].copy()

    print(
        f"Событий после соединения: {before_filter:,}; "
        f"допустимых до assignment_ts: {len(merged):,}"
    )

    merged["event_age_hours"] = (
        merged["assignment_ts"] - merged["event_ts"]
    ).dt.total_seconds() / 3600.0
    merged["event_age_days"] = merged["event_age_hours"] / 24.0
    merged["event_day"] = merged["event_ts"].dt.floor("D")

    feature_parts = []
    all_leads = assignment_frame[["lead_id"]].drop_duplicates().set_index("lead_id")

    # Общие агрегаты за всю доступную историю до назначения
    grouped = merged.groupby("lead_id", sort=False)
    general = pd.DataFrame(index=grouped.size().index)
    general["ev_count_all"] = grouped.size()
    general["ev_recency_hours"] = grouped["event_age_hours"].min()
    general["ev_history_span_hours"] = (
        grouped["event_ts"].max() - grouped["event_ts"].min()
    ).dt.total_seconds() / 3600.0
    general["ev_type_nunique"] = grouped["event_type"].nunique()
    general["ev_active_days_all"] = grouped["event_day"].nunique()

    if "ctx_seq" in merged.columns:
        general["ev_ctx_seq_nunique"] = grouped["ctx_seq"].nunique()

    if "src_slot" in merged.columns:
        general["ev_src_slot_nunique"] = grouped["src_slot"].nunique()
        general["ev_src_slot_mean"] = grouped["src_slot"].mean()
        general["ev_src_slot_std"] = grouped["src_slot"].std()
        general["ev_src_slot_min"] = grouped["src_slot"].min()
        general["ev_src_slot_max"] = grouped["src_slot"].max()

    if "item_price_log" in merged.columns:
        general["ev_item_price_mean"] = grouped["item_price_log"].mean()
        general["ev_item_price_std"] = grouped["item_price_log"].std()
        general["ev_item_price_min"] = grouped["item_price_log"].min()
        general["ev_item_price_max"] = grouped["item_price_log"].max()
        general["ev_item_price_last"] = (
            merged.sort_values("event_ts")
            .groupby("lead_id", sort=False)["item_price_log"]
            .last()
        )

    feature_parts.append(general)

    # Агрегаты по фиксированным окнам
    for window in windows:
        recent = merged.loc[
            (merged["event_age_days"] >= 0)
            & (merged["event_age_days"] <= window)
        ].copy()

        window_features = pd.DataFrame(
            index=recent["lead_id"].drop_duplicates()
        )

        window_features[f"ev_count_{window}d"] = (
            recent.groupby("lead_id", sort=False).size()
        )
        window_features[f"ev_active_days_{window}d"] = (
            recent.groupby("lead_id", sort=False)["event_day"].nunique()
        )

        type_counts = (
            recent.groupby(["lead_id", "event_type"], sort=False)
            .size()
            .unstack(fill_value=0)
        )

        for event_type in event_types:
            if event_type in type_counts.columns:
                window_features[f"ev_{event_type}_count_{window}d"] = (
                    type_counts[event_type]
                )
            else:
                window_features[f"ev_{event_type}_count_{window}d"] = 0

        if "ctx_seq" in recent.columns:
            window_features[f"ev_ctx_seq_nunique_{window}d"] = (
                recent.groupby("lead_id", sort=False)["ctx_seq"].nunique()
            )

        if "src_slot" in recent.columns:
            window_features[f"ev_src_slot_mean_{window}d"] = (
                recent.groupby("lead_id", sort=False)["src_slot"].mean()
            )

        feature_parts.append(window_features)

    # Recency отдельно для каждого типа события
    recency = (
        merged.groupby(["lead_id", "event_type"], sort=False)["event_age_hours"]
        .min()
        .unstack()
    )
    recency = recency.rename(
        columns={
            event_type: f"ev_{event_type}_recency_hours"
            for event_type in recency.columns
        }
    )
    feature_parts.append(recency)

    event_features = pd.concat(feature_parts, axis=1)
    event_features = all_leads.join(event_features, how="left")

    # Отсутствие события — осмысленное состояние:
    # счётчики заполняются нулями, recency/statistics остаются NaN.
    count_columns = [
        col for col in event_features.columns
        if "_count_" in col
        or col.startswith("ev_count_")
        or "active_days" in col
        or col.endswith("_nunique")
        or "_nunique_" in col
    ]
    event_features[count_columns] = event_features[count_columns].fillna(0)

    event_features = event_features.reset_index()
    return event_features


: 

In [ ]:

# Признаки из events строятся сразу для train и test.
# Это не использует target и не смешивает будущие события:
# для каждой строки действует индивидуальный фильтр event_ts < assignment_ts.
all_assignments = pd.concat(
    [
        train_raw.drop(columns=["target"]),
        test_raw,
    ],
    axis=0,
    ignore_index=True,
)

event_features = make_event_features(all_assignments, events_raw)
print("Event features:", event_features.shape)

train_features = (
    make_tabular_features(train_raw)
    .merge(event_features, on="lead_id", how="left", validate="one_to_one")
)
test_features = (
    make_tabular_features(test_raw)
    .merge(event_features, on="lead_id", how="left", validate="one_to_one")
)

assert len(train_features) == len(train_raw)
assert len(test_features) == len(test_raw)

print("Train after feature engineering:", train_features.shape)
print("Test after feature engineering:", test_features.shape)


Событий после соединения: 254,705; допустимых до assignment_ts: 238,581
Event features: (18000, 76)
Train after feature engineering: (13694, 385)
Test after feature engineering: (4306, 384)


: 

## 4. Финальная обработка признаков

In [ ]:

TARGET = "target"

# Сырые datetime исключаются после извлечения календарных признаков
DROP_COLUMNS = [
    TARGET,
    "lead_id",
    "assignment_ts",
    "assignment_date",
]

feature_columns = [
    column
    for column in train_features.columns
    if column not in DROP_COLUMNS
]

assert set(feature_columns) == (
    set(test_features.columns) - {"lead_id", "assignment_ts", "assignment_date"}
)

X_all = train_features[feature_columns].copy()
X_test = test_features[feature_columns].copy()
y_all = train_features[TARGET].astype("int8").to_numpy()
assignment_dates = pd.to_datetime(train_features["assignment_date"])

categorical_columns = [
    column
    for column in feature_columns
    if (
        X_all[column].dtype == "object"
        or str(X_all[column].dtype).startswith("string")
        or str(X_all[column].dtype) == "category"
    )
]

numeric_columns = [
    column for column in feature_columns
    if column not in categorical_columns
]

# Нормализация типов
for column in categorical_columns:
    X_all[column] = X_all[column].fillna("__MISSING__").astype(str)
    X_test[column] = X_test[column].fillna("__MISSING__").astype(str)

for column in numeric_columns:
    X_all[column] = (
        pd.to_numeric(X_all[column], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .astype("float32")
    )
    X_test[column] = (
        pd.to_numeric(X_test[column], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .astype("float32")
    )

# Удаляем константные признаки по train
constant_columns = [
    column for column in feature_columns
    if X_all[column].nunique(dropna=False) <= 1
]

if constant_columns:
    X_all = X_all.drop(columns=constant_columns)
    X_test = X_test.drop(columns=constant_columns)
    feature_columns = [
        column for column in feature_columns
        if column not in constant_columns
    ]
    categorical_columns = [
        column for column in categorical_columns
        if column not in constant_columns
    ]
    numeric_columns = [
        column for column in numeric_columns
        if column not in constant_columns
    ]

assert list(X_all.columns) == list(X_test.columns)

print("Final features:", len(feature_columns))
print("Categorical:", len(categorical_columns))
print("Numeric:", len(numeric_columns))
print("Removed constants:", len(constant_columns))


Final features: 373
Categorical: 0
Numeric: 373
Removed constants: 8


: 

## 5. Временная validation-выборка

In [ ]:

# Последние 4 уникальные даты train используются как validation
unique_dates = np.sort(assignment_dates.dt.normalize().unique())

VALID_DAYS = 4
assert len(unique_dates) > VALID_DAYS

validation_dates = unique_dates[-VALID_DAYS:]
validation_start = pd.Timestamp(validation_dates[0])

train_mask = assignment_dates.dt.normalize() < validation_start
valid_mask = ~train_mask

X_train = X_all.loc[train_mask].reset_index(drop=True)
X_valid = X_all.loc[valid_mask].reset_index(drop=True)
y_train = y_all[train_mask.to_numpy()]
y_valid = y_all[valid_mask.to_numpy()]
dates_valid = assignment_dates.loc[valid_mask].reset_index(drop=True)

print("Train period:", assignment_dates.loc[train_mask].min(), "—", assignment_dates.loc[train_mask].max())
print("Valid period:", dates_valid.min(), "—", dates_valid.max())
print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)
print("Train target rate:", y_train.mean())
print("Valid target rate:", y_valid.mean())

valid_target_by_day = (
    pd.DataFrame({"date": dates_valid.dt.normalize(), "target": y_valid})
    .groupby("date")
    .agg(rows=("target", "size"), positives=("target", "sum"), rate=("target", "mean"))
)
display(valid_target_by_day)


Train period: 2026-04-07 00:00:00 — 2026-04-18 00:00:00
Valid period: 2026-04-19 00:00:00 — 2026-04-22 00:00:00
Train shape: (10272, 373)
Valid shape: (3422, 373)
Train target rate: 0.20813862928348908
Valid target rate: 0.2054354178842782


,rows,positives,rate
date,,,
2026-04-19,863,171,0.198146
2026-04-20,861,165,0.191638
2026-04-21,837,197,0.235364
2026-04-22,861,170,0.197445


: 

## 6. CatBoost

In [ ]:

catboost_params = {
    "iterations": 1400,
    "depth": 7,
    "learning_rate": 0.035,
    "loss_function": "Logloss",
    "eval_metric": "PRAUC",
    "l2_leaf_reg": 7.0,
    "random_strength": 0.5,
    "subsample": 0.85,
    "bootstrap_type": "Bernoulli",
    "random_seed": SEED,
    "thread_count": -1,
    "verbose": 100,
    "allow_writing_files": False,
}

catboost_model = CatBoostClassifier(**catboost_params)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=categorical_columns,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=150,
    use_best_model=True,
)

pred_valid_catboost = catboost_model.predict_proba(X_valid)[:, 1]

catboost_daily_ap, catboost_per_day = daily_average_precision(
    y_valid,
    pred_valid_catboost,
    dates_valid,
)
catboost_global_ap = average_precision_score(y_valid, pred_valid_catboost)

print("CatBoost best iteration:", catboost_model.get_best_iteration())
print("CatBoost Daily AP:", catboost_daily_ap)
print("CatBoost global AP:", catboost_global_ap)
display(catboost_per_day)


0:	learn: 0.5159931	test: 0.5134440	best: 0.5134440 (0)	total: 14.7ms	remaining: 20.6s
100:	learn: 0.6942193	test: 0.6354417	best: 0.6354417 (100)	total: 812ms	remaining: 10.4s
200:	learn: 0.7717925	test: 0.6531920	best: 0.6531920 (200)	total: 1.54s	remaining: 9.19s
300:	learn: 0.8413391	test: 0.6608981	best: 0.6608981 (300)	total: 2.29s	remaining: 8.35s
400:	learn: 0.8985470	test: 0.6666492	best: 0.6667409 (391)	total: 3.03s	remaining: 7.56s
500:	learn: 0.9387647	test: 0.6712458	best: 0.6713553 (498)	total: 3.79s	remaining: 6.79s
600:	learn: 0.9651612	test: 0.6725275	best: 0.6726816 (585)	total: 4.52s	remaining: 6.01s
700:	learn: 0.9811758	test: 0.6732099	best: 0.6736880 (674)	total: 5.25s	remaining: 5.24s
800:	learn: 0.9906439	test: 0.6739672	best: 0.6741335 (742)	total: 6.02s	remaining: 4.5s
900:	learn: 0.9955468	test: 0.6743948	best: 0.6746988 (891)	total: 6.77s	remaining: 3.75s
1000:	learn: 0.9982721	test: 0.6748188	best: 0.6751115 (978)	total: 7.51s	remaining: 2.99s
1100:	learn: 

,date,n_rows,n_positive,positive_rate,average_precision
0,2026-04-19,863,171,0.198146,0.722979
1,2026-04-20,861,165,0.191638,0.638270
2,2026-04-21,837,197,0.235364,0.638442
3,2026-04-22,861,170,0.197445,0.716222


: 

## 7. LightGBM

In [ ]:

def prepare_lightgbm_categories(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    categorical_features,
):
    '''
    Создаёт одинаковый набор категорий для train/valid/test.
    Используются только сами значения категорий, без target.
    '''
    train_out = train_df.copy()
    valid_out = valid_df.copy()
    test_out = test_df.copy()

    for column in categorical_features:
        categories = pd.Index(
            pd.concat(
                [
                    train_out[column],
                    valid_out[column],
                    test_out[column],
                ],
                axis=0,
                ignore_index=True,
            ).astype(str).unique()
        )

        dtype = pd.CategoricalDtype(categories=categories, ordered=False)
        train_out[column] = train_out[column].astype(dtype)
        valid_out[column] = valid_out[column].astype(dtype)
        test_out[column] = test_out[column].astype(dtype)

    return train_out, valid_out, test_out

X_train_lgb, X_valid_lgb, X_test_lgb = prepare_lightgbm_categories(
    X_train,
    X_valid,
    X_test,
    categorical_columns,
)

lightgbm_params = {
    "objective": "binary",
    "n_estimators": 3000,
    "learning_rate": 0.02,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 80,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.5,
    "reg_lambda": 4.0,
    "max_bin": 255,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
}

lightgbm_model = LGBMClassifier(**lightgbm_params)

lightgbm_model.fit(
    X_train_lgb,
    y_train,
    eval_set=[(X_valid_lgb, y_valid)],
    eval_metric="average_precision",
    categorical_feature=categorical_columns,
    callbacks=[
        early_stopping(stopping_rounds=200, verbose=True),
        log_evaluation(period=100),
    ],
)

pred_valid_lightgbm = lightgbm_model.predict_proba(
    X_valid_lgb,
    num_iteration=lightgbm_model.best_iteration_,
)[:, 1]

lightgbm_daily_ap, lightgbm_per_day = daily_average_precision(
    y_valid,
    pred_valid_lightgbm,
    dates_valid,
)
lightgbm_global_ap = average_precision_score(y_valid, pred_valid_lightgbm)

print("LightGBM best iteration:", lightgbm_model.best_iteration_)
print("LightGBM Daily AP:", lightgbm_daily_ap)
print("LightGBM global AP:", lightgbm_global_ap)
display(lightgbm_per_day)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's average_precision: 0.636153	valid_0's binary_logloss: 0.388504
[200]	valid_0's average_precision: 0.665776	valid_0's binary_logloss: 0.364554
[300]	valid_0's average_precision: 0.676454	valid_0's binary_logloss: 0.357055
[400]	valid_0's average_precision: 0.683245	valid_0's binary_logloss: 0.352386
[500]	valid_0's average_precision: 0.687466	valid_0's binary_logloss: 0.349211
[600]	valid_0's average_precision: 0.690333	valid_0's binary_logloss: 0.347365
[700]	valid_0's average_precision: 0.692915	valid_0's binary_logloss: 0.346244
[800]	valid_0's average_precision: 0.693651	valid_0's binary_logloss: 0.346303
[900]	valid_0's average_precision: 0.694438	valid_0's binary_logloss: 0.346582
Early stopping, best iteration is:
[730]	valid_0's average_precision: 0.693471	valid_0's binary_logloss: 0.345989
LightGBM best iteration: 730
LightGBM Daily AP: 0.6958004484806293
LightGBM global AP: 0.6934713200026361


,date,n_rows,n_positive,positive_rate,average_precision
0,2026-04-19,863,171,0.198146,0.733502
1,2026-04-20,861,165,0.191638,0.662796
2,2026-04-21,837,197,0.235364,0.651937
3,2026-04-22,861,170,0.197445,0.734967


: 

## 8. Подбор веса ансамбля

In [ ]:

ensemble_rows = []

# weight_catboost = 0 означает чистый LightGBM,
# weight_catboost = 1 означает чистый CatBoost.
for weight_catboost in np.linspace(0.0, 1.0, 41):
    blended = (
        weight_catboost * pred_valid_catboost
        + (1.0 - weight_catboost) * pred_valid_lightgbm
    )

    daily_ap_value, _ = daily_average_precision(
        y_valid,
        blended,
        dates_valid,
    )
    global_ap_value = average_precision_score(y_valid, blended)

    ensemble_rows.append({
        "weight_catboost": float(weight_catboost),
        "weight_lightgbm": float(1.0 - weight_catboost),
        "daily_ap": daily_ap_value,
        "global_ap": global_ap_value,
    })

ensemble_search = (
    pd.DataFrame(ensemble_rows)
    .sort_values(["daily_ap", "global_ap"], ascending=False)
    .reset_index(drop=True)
)

best_weight_catboost = float(ensemble_search.loc[0, "weight_catboost"])
best_weight_lightgbm = 1.0 - best_weight_catboost

pred_valid_ensemble = (
    best_weight_catboost * pred_valid_catboost
    + best_weight_lightgbm * pred_valid_lightgbm
)

ensemble_daily_ap, ensemble_per_day = daily_average_precision(
    y_valid,
    pred_valid_ensemble,
    dates_valid,
)
ensemble_global_ap = average_precision_score(
    y_valid,
    pred_valid_ensemble,
)

print("Best CatBoost weight:", best_weight_catboost)
print("Best LightGBM weight:", best_weight_lightgbm)
display(ensemble_search.head(10))


Best CatBoost weight: 0.1
Best LightGBM weight: 0.9


,weight_catboost,weight_lightgbm,daily_ap,global_ap
0,0.100,0.900,0.696352,0.694102
1,0.075,0.925,0.696315,0.693956
2,0.150,0.850,0.696272,0.693983
3,0.200,0.800,0.696236,0.694014
4,0.175,0.825,0.696205,0.693943
5,0.225,0.775,0.696177,0.693967
6,0.275,0.725,0.696148,0.693920
7,0.125,0.875,0.696130,0.693981
8,0.250,0.750,0.696121,0.693900
9,0.050,0.950,0.696077,0.693777


: 

In [ ]:

metrics_table = pd.DataFrame([
    {
        "model": "CatBoost",
        "daily_average_precision": catboost_daily_ap,
        "global_average_precision": catboost_global_ap,
    },
    {
        "model": "LightGBM",
        "daily_average_precision": lightgbm_daily_ap,
        "global_average_precision": lightgbm_global_ap,
    },
    {
        "model": (
            f"Ensemble: {best_weight_catboost:.3f} CatBoost + "
            f"{best_weight_lightgbm:.3f} LightGBM"
        ),
        "daily_average_precision": ensemble_daily_ap,
        "global_average_precision": ensemble_global_ap,
    },
]).sort_values("daily_average_precision", ascending=False)

display(metrics_table)
display(ensemble_per_day)

metrics_table.to_csv("validation_metrics.csv", index=False)
ensemble_per_day.to_csv("validation_daily_ap.csv", index=False)


,model,daily_average_precision,global_average_precision
2,Ensemble: 0.100 CatBoost + 0.900 LightGBM,0.696352,0.694102
1,LightGBM,0.695800,0.693471
0,CatBoost,0.678978,0.677409


,date,n_rows,n_positive,positive_rate,average_precision
0,2026-04-19,863,171,0.198146,0.734920
1,2026-04-20,861,165,0.191638,0.661935
2,2026-04-21,837,197,0.235364,0.652805
3,2026-04-22,861,170,0.197445,0.735747


: 

## 9. Финальное обучение на всём train

In [ ]:

# Для финального обучения используем число итераций, найденное early stopping на временной validation
catboost_best_iterations = max(
    50,
    int(catboost_model.get_best_iteration()) + 1,
)
lightgbm_best_iterations = max(
    50,
    int(lightgbm_model.best_iteration_),
)

print("Final CatBoost iterations:", catboost_best_iterations)
print("Final LightGBM iterations:", lightgbm_best_iterations)

final_catboost_params = catboost_params.copy()
final_catboost_params["iterations"] = catboost_best_iterations
final_catboost_params["verbose"] = 100

final_catboost = CatBoostClassifier(**final_catboost_params)
final_catboost.fit(
    X_all,
    y_all,
    cat_features=categorical_columns,
)

# Категории LightGBM для полного train и test
X_all_lgb = X_all.copy()
X_test_final_lgb = X_test.copy()

for column in categorical_columns:
    categories = pd.Index(
        pd.concat(
            [X_all_lgb[column], X_test_final_lgb[column]],
            axis=0,
            ignore_index=True,
        ).astype(str).unique()
    )
    dtype = pd.CategoricalDtype(categories=categories, ordered=False)
    X_all_lgb[column] = X_all_lgb[column].astype(dtype)
    X_test_final_lgb[column] = X_test_final_lgb[column].astype(dtype)

final_lightgbm_params = lightgbm_params.copy()
final_lightgbm_params["n_estimators"] = lightgbm_best_iterations

final_lightgbm = LGBMClassifier(**final_lightgbm_params)
final_lightgbm.fit(
    X_all_lgb,
    y_all,
    categorical_feature=categorical_columns,
    callbacks=[log_evaluation(period=100)],
)

pred_test_catboost = final_catboost.predict_proba(X_test)[:, 1]
pred_test_lightgbm = final_lightgbm.predict_proba(
    X_test_final_lgb
)[:, 1]

pred_test_ensemble = (
    best_weight_catboost * pred_test_catboost
    + best_weight_lightgbm * pred_test_lightgbm
)

assert np.isfinite(pred_test_ensemble).all()
assert ((pred_test_ensemble >= 0) & (pred_test_ensemble <= 1)).all()


Final CatBoost iterations: 1194
Final LightGBM iterations: 730
0:	learn: 0.4981521	total: 24.3ms	remaining: 29s
100:	learn: 0.6857176	total: 893ms	remaining: 9.66s
200:	learn: 0.7529829	total: 1.73s	remaining: 8.54s
300:	learn: 0.8109615	total: 2.55s	remaining: 7.56s
400:	learn: 0.8590548	total: 3.36s	remaining: 6.65s
500:	learn: 0.8995791	total: 4.18s	remaining: 5.78s
600:	learn: 0.9309719	total: 5s	remaining: 4.93s
700:	learn: 0.9543933	total: 5.84s	remaining: 4.11s
800:	learn: 0.9699052	total: 6.67s	remaining: 3.27s
900:	learn: 0.9821350	total: 7.5s	remaining: 2.44s
1000:	learn: 0.9892569	total: 8.31s	remaining: 1.6s
1100:	learn: 0.9938244	total: 9.13s	remaining: 771ms
1193:	learn: 0.9966450	total: 9.91s	remaining: 0us


: 

## 10. Submission и проверки воспроизводимости

In [ ]:

submission = pd.DataFrame({
    "lead_id": test_raw["lead_id"].to_numpy(),
    "score": pred_test_ensemble,
})

# Строго две требуемые колонки
submission = submission[["lead_id", "score"]]

assert len(submission) == len(test_raw)
assert submission["lead_id"].is_unique
assert submission["score"].notna().all()
assert list(submission.columns) == ["lead_id", "score"]

# Проверяем порядок относительно sample_submission
if "lead_id" in sample_submission.columns:
    assert submission["lead_id"].tolist() == sample_submission["lead_id"].tolist()

submission.to_csv("submission.csv", index=False)

print(submission.shape)
display(submission.head())
print("Создан файл:", Path("submission.csv").resolve())


(4306, 2)


,lead_id,score
0,lead_97e409eb8f8c8246,0.112644
1,lead_55310edb4489f9e9,0.104925
2,lead_e7f653a2c6a7eee8,0.870471
3,lead_22f8e1cfc487ac20,0.021606
4,lead_48b638b839abfac3,0.110490


Создан файл: /Users/timurkudaev/Downloads/lead_prioritization_challenge/submission.csv


: 

In [ ]:

# Сохраняем параметры запуска и список признаков
run_info = {
    "seed": SEED,
    "validation_days": VALID_DAYS,
    "validation_start": str(validation_start),
    "n_features": len(feature_columns),
    "categorical_columns": categorical_columns,
    "constant_columns_removed": constant_columns,
    "catboost_best_iterations": catboost_best_iterations,
    "lightgbm_best_iterations": lightgbm_best_iterations,
    "ensemble_weight_catboost": best_weight_catboost,
    "ensemble_weight_lightgbm": best_weight_lightgbm,
    "catboost_params": final_catboost_params,
    "lightgbm_params": final_lightgbm_params,
}

with open("run_info.json", "w", encoding="utf-8") as file:
    json.dump(run_info, file, ensure_ascii=False, indent=2, default=str)

with open("feature_columns.json", "w", encoding="utf-8") as file:
    json.dump(feature_columns, file, ensure_ascii=False, indent=2)

print("Сохранены:")
print("- submission.csv")
print("- validation_metrics.csv")
print("- validation_daily_ap.csv")
print("- run_info.json")
print("- feature_columns.json")


Сохранены:
- submission.csv
- validation_metrics.csv
- validation_daily_ap.csv
- run_info.json
- feature_columns.json


: 


## Что особенно важно в обработке признаков

- `lead_id` исключён из модели как уникальный идентификатор.
- `user_id` оставлен как категориальный признак: повторяющиеся пользователи могут нести исторический сигнал, а CatBoost и LightGBM умеют работать с неизвестными/редкими категориями.
- Категориальные пропуски заменяются отдельным значением `__MISSING__`.
- Числовые `inf/-inf` переводятся в `NaN`; обе модели умеют обрабатывать числовые пропуски.
- Категории LightGBM задаются единым `CategoricalDtype`, поэтому коды категорий совпадают между train, validation и test.
- Из `events.csv` не используется ни одного события после назначения.
- Target encoding и другие target-агрегаты намеренно не добавлены: при коротком временном диапазоне они создают высокий риск переобучения и утечки.
- Вес ансамбля выбирается только по временной validation и по основной метрике Daily AP.
